# Inventario de datos de Olist (E-Commerce)

Este notebook arma un mapa general del dataset de Olist antes de entrar en
cualquier misión específica. No responde ninguna pregunta de negocio todavía:
documenta qué tablas hay, qué tamaño tienen, cómo se conectan entre sí y dónde
hay valores nulos. Cada misión (recomendaciones, entregas, sentimiento) parte de
este conocimiento base y hace su propio análisis exploratorio enfocado en su
propia pregunta.

In [1]:
import pandas as pd
from pathlib import Path

data_raw = Path("..") / "data" / "raw"

# Mapa: nombre simbolico de cada tabla -> nombre real del CSV
archivos = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

# Carga todas las tablas en un dict nombre -> DataFrame
tablas = {nombre: pd.read_csv(data_raw / archivo) for nombre, archivo in archivos.items()}

## Recuento de tablas y su tamaño

Cuántas filas y columnas tiene cada tabla. Esto da una primera noción de la escala
de cada una: por ejemplo, order_items debería tener más filas que orders, porque un
pedido puede tener varios items.

In [2]:
# Dimensiones de cada tabla
shapes = {nombre: df.shape for nombre, df in tablas.items()}

# Transf. el dict a df para leerlo en filas (una por tabla)
pd.DataFrame(shapes, index=["filas", "columnas"]).T

,filas,columnas
customers,99441,5
geolocation,1000163,5
orders,99441,8
order_items,112650,7
order_payments,103886,5
order_reviews,99224,7
products,32951,9
sellers,3095,4
category_translation,71,2


## Mapa de relaciones entre tablas

El dataset está normalizado: cada tabla cubre una entidad, y se conectan entre sí
por claves compartidas.

| Tabla A | Clave | Tabla B |
|---|---|---|
| orders | customer_id | customers |
| orders | order_id | order_items |
| order_items | product_id | products |
| order_items | seller_id | sellers |
| orders | order_id | order_payments |
| orders | order_id | order_reviews |
| products | product_category_name | category_translation |
| customers | customer_zip_code_prefix | geolocation (zip_code_prefix) |
| sellers | seller_zip_code_prefix | geolocation (zip_code_prefix) |

orders es la tabla central: casi todo el resto se conecta a ella directa o
indirectamente a través de order_id o customer_id.

## Perfil de tipos y valores faltantes

Para cada tabla: cuántas columnas hay de cada tipo de dato, y qué porcentaje de
valores nulos tiene cada columna. Esto anticipa trabajo de limpieza que cada
misión va a tener que resolver por su cuenta.

In [3]:
# Recorre cada tabla y contruye un resumen (tipo de dato y porc. de nulos)
for nombre, df in tablas.items():
    resumen = pd.DataFrame({ 
        "dtype": df.dtypes,
        "pct_nulos": (df.isna().mean() * 100).round(2),
    })

    print(f"--- {nombre} ---")
    print(resumen)
    print()

--- customers ---
                          dtype  pct_nulos
customer_id                 str        0.0
customer_unique_id          str        0.0
customer_zip_code_prefix  int64        0.0
customer_city               str        0.0
customer_state              str        0.0

--- geolocation ---
                               dtype  pct_nulos
geolocation_zip_code_prefix    int64        0.0
geolocation_lat              float64        0.0
geolocation_lng              float64        0.0
geolocation_city                 str        0.0
geolocation_state                str        0.0

--- orders ---
                              dtype  pct_nulos
order_id                        str       0.00
customer_id                     str       0.00
order_status                    str       0.00
order_purchase_timestamp        str       0.00
order_approved_at               str       0.16
order_delivered_carrier_date    str       1.79
order_delivered_customer_date   str       2.98
order_estimated_deliver

--- products ---
                              dtype  pct_nulos
product_id                      str       0.00
product_category_name           str       1.85
product_name_lenght         float64       1.85
product_description_lenght  float64       1.85
product_photos_qty          float64       1.85
product_weight_g            float64       0.01
product_length_cm           float64       0.01
product_height_cm           float64       0.01
product_width_cm            float64       0.01

--- sellers ---
                        dtype  pct_nulos
seller_id                 str        0.0
seller_zip_code_prefix  int64        0.0
seller_city               str        0.0
seller_state              str        0.0

--- category_translation ---
                              dtype  pct_nulos
product_category_name           str        0.0
product_category_name_english   str        0.0

